<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Was sind KI-Agenten? Tool Use & Function Calling
</b></font> </br>

---

**Leitprojekt:** Der gesamte Kurs baut auf einem durchgehenden Anwendungsfall auf — dem **Meeting- & Research-Briefing-Agent**, einem kontrollierten Agentensystem, das Projektunterlagen und Meeting-Protokolle sichtet, Aussagen mit Quellen belegt und bei Unsicherheit eskaliert.



**Beitrag zum Leitprojekt:**    
M01 legt die begriffliche Grundlage und die erste kontrollierte Handlungsebene. Die Progression von einfachem zu kooperierendem Agenten ist genau der Weg, den der Meeting- & Research-Briefing-Agent im Kursverlauf durchläuft — von "Such-Tool mit nachvollziehbaren Entscheidungen" bis zu "HITL, Memory und spezialisierten Workern". Bevor der Agent Protokolle und Unterlagen sichtet, muss er planen, welche Fähigkeiten ihm zur Verfügung stehen, und anschließend gezielt handeln — die Tool-Schnittstelle macht sichtbar, was erlaubt, beobachtbar und prüfbar ist.

In [ ]:
#@title  🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os


from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
)
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)


print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS


# 1 | Was ist ein KI-Agent?
---

Wenn man sich mit generativer KI beschäftigt, stößt man früher oder später auf den Begriff **Agent** – also ein System, das Aufgaben eigenständig ausführt. Doch was genau ist ein *„echter“* Agent? Muss er vollständig autonom sein?



<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/agentisch_2.png" class="logo" width="900"/>
<p><font color='black' size="2">
KI-generiertes Bild
</font></p>




Besonders einfach ist der Einstieg bei eher einfachen Aufgaben – also bei Prozessen, die heute noch manuell erledigt werden, wie das Ausfüllen von Formularen, das Nachschlagen in einer Datenbank oder das Kopieren von Informationen zwischen verschiedenen Systemen. Diese Aufgaben lassen sich gut in sogenannte agentische Workflows überführen – also in Abläufe, bei denen die KI (teilweise) selbstständig handelt.


Natürlich gibt es auch deutlich komplexere Anwendungen, bei denen die KI viele Entscheidungen trifft, Schleifen durchläuft und sich an neue Situationen anpasst. Solche Systeme sind spannend – aber gerade für den Anfang ist es oft sinnvoller, mit kleineren, überschaubaren Schritten zu starten. Dort liegen aktuell auch die meisten Chancen, GenAI im Alltag oder im Beruf sinnvoll einzusetzen.



LLM's und KI-Agenten bauen aufeinander auf, sind aber nicht dasselbe.

| Ebene | Beschreibung | Typisches Ergebnis |
|---|---|---|
| **LLM-Call** | Ein Prompt, eine Antwort | Textantwort |
| **GenAI-App** | Prompt + UI + einfache Logik | Chatbot, Assistent |
| **KI-Agent** | LLM + Tool-Nutzung + Entscheidungslogik + Schleife | Mehrstufige Aufgabenerfüllung |




**Wann reicht ein LLM-Call?**
- Reines Umformulieren, Zusammenfassen, Übersetzen
- Keine externen Datenquellen oder Aktionen nötig

**Wann lohnt ein Agent?**
- Mehrere Schritte mit Zwischenentscheidungen
- Externe Aktionen (APIs, Datenbanken, Dateien)
- Rückfragen, Fehlerbehandlung und Zielverfolgung

# 2 | Die 5 Eigenschaften eines Agenten
---

Ein praxisreifer Agent zeigt fünf Kerneigenschaften:

1. **Autonomie**: kann Teilentscheidungen selbst treffen.

2. **Reaktionsfähigkeit**: reagiert auf neue Informationen.

3. **Proaktivität**: verfolgt ein Ziel über mehrere Schritte.

4. ***"Soziale"*** **Fähigkeit**: interagiert klar mit Mensch und/oder anderen Agenten.

5. **Handlungsfähigkeit**: kann Werkzeuge, APIs oder Systeme aktiv einsetzen, um Schritte umzusetzen.

Je mehr davon robust umgesetzt sind, desto eher spricht man von einem Agenten statt einem einfachen Sprachmodell bzw. Chat-Interface.

# 3 | ReAct und verbreitete Agenten-Ansätze
---

Ein KI-Agent löst Aufgaben häufig nicht in einem einzigen Schritt. Stattdessen kann er in einer **Schleife** arbeiten: Er entscheidet, was als Nächstes zu tun ist, führt eine Aktion aus – zum Beispiel den Aufruf eines Tools –, betrachtet das Ergebnis und entscheidet auf dieser Basis über den nächsten Schritt.

Ein verbreitetes Muster für eine solche Agentenschleife ist **ReAct** (kurz für *Reasoning + Acting*).

**Beispiel**

```text
Gedanke:     Ich brauche das aktuelle Wetter in Essen.
Aktion:      weather("Essen")
Beobachtung: 22 °C, leichter Regen
Gedanke:     Jetzt kann ich die Frage beantworten.
Antwort:     ...
```

Der Agent kann diesen Zyklus so lange wiederholen, bis er die Aufgabe lösen kann. Dabei nutzt er das Ergebnis jeder Aktion (die *Observation*), um seine nächste Entscheidung zu treffen.

Vereinfacht lässt sich der Ablauf so darstellen:

**Entscheiden/Planen → Tool aufrufen → Ergebnis beobachten → nächsten Schritt bestimmen**

> **Wichtig:** Der „Gedanke“ muss nicht als offen sichtbarer Denkprozess des Modells verstanden werden. In praktischen Systemen werden typischerweise die Aktionen, Tool-Aufrufe, Ergebnisse und gegebenenfalls explizit erzeugte Pläne sichtbar gemacht – nicht die interne Gedankenkette des Modells.

ReAct ist ein etablierter Fachbegriff und begegnet in der Fachliteratur sowie bei verschiedenen Agenten-Frameworks. Es ist jedoch **nicht die einzige mögliche Agenten-Architektur**.

**Roter Faden: Planen – Handeln – Prüfen**

- **Planen:** Der Agent ordnet die Anfrage ein und wählt eine passende Fähigkeit oder ein Tool.
- **Handeln:** Das ausgewählte Tool wird mit validierten Parametern ausgeführt.
- **Prüfen:** Das Ergebnis wird auf Fehler, ausreichenden Kontext und Unsicherheit geprüft. Danach antwortet der Agent, fragt nach oder stoppt.

M01 legt vor allem die Handlungsebene mit Tools offen. M02 macht die Tool-Auswahl durch `create_agent()` ausführbar; M03 behandelt die Prompt-Regeln für kontrolliertes Verhalten.

# 4 |  Von einfachen Systemen zu mehr Autonomie
---




Aufbauend auf solchen Grundprinzipien gibt es unterschiedliche Agenten-Ansätze und Erweiterungen. Sie geben einem System unterschiedlich viel Freiheit bei der Auswahl von Aktionen, der Planung, der Selbstkontrolle und der Koordination.

Die folgende Übersicht zeigt die verschiedenen Ansätze vereinfacht und mit tendenziell zunehmender Autonomie:



| Stufe | Ansatz                        | Autonomie            | Kernidee                                                                               |
| ----: | ----------------------------- | -------------------- | -------------------------------------------------------------------------------------- |
|     1 | **Einfache reaktive Systeme** | Sehr gering          | Reagieren nach festen Regeln oder Zuständen; kein LLM-basiertes Planen                 |
|     2 | **Function Calling**          | Gering               | Das Modell wählt ein passendes Tool und dessen Parameter                               |
|     3 | **ReAct**                     | Mittel               | Wiederholt Entscheiden → Handeln → Beobachten und bestimmt daraus den nächsten Schritt |
|     4 | **Plan-and-Execute**          | Hoch                 | Erstellt zunächst einen mehrstufigen Plan und führt ihn anschließend aus               |
|     5 | **ReAct + Reflexion**         | Sehr hoch            | Prüft eigene Ergebnisse und entscheidet über Korrekturen oder weitere Schritte         |
|     6 | **Multi-Agent-Systeme**       | Potenziell sehr hoch | Mehrere spezialisierte Agenten teilen Aufgaben auf und koordinieren ihre Ergebnisse    |

Die Ansätze schließen sich dabei **nicht gegenseitig aus**. Ein Agent kann beispielsweise ReAct mit einem Planungsmechanismus und einer Reflexionsphase kombinieren. Ebenso kann ein Multi-Agent-System mehrere ReAct-Agenten enthalten.

Auch **Reasoning-Modelle** bilden keine eigene Stufe dieser Tabelle. Sie beschreiben eher eine Fähigkeit bzw. Modellklasse und können als Grundlage verschiedener Agenten-Architekturen eingesetzt werden.

Als vereinfachte Entwicklung lässt sich festhalten:

**Regeln → Tool-Auswahl → iterative Tool-Nutzung → Planung → Selbstkorrektur → Koordination**

Mit zunehmender Autonomie steigen allerdings typischerweise auch **Komplexität, Kosten, Laufzeit und Fehlerrisiko**. Mehr Autonomie bedeutet daher nicht automatisch ein besseres System. Für viele Aufgaben ist eine möglichst einfache, kontrollierbare Agenten-Architektur die sinnvollere Lösung.


<p><font color='darkblue' size="4">
 <b>✨ Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)


# 5 | LangChain 101
---

## 5.1 | Was ist LangChain?

LangChain ist ein Framework zur **Entwicklung von Anwendungen mit großen Sprachmodellen** (LLMs). Es stellt modulare Bausteine bereit — für einzelne LLM-Aufrufe, für komplexe Workflows, für die Einbindung externer Datenquellen und für Konversationsgedächtnis. Die zentrale Idee: Bausteine wie Prompt, Modell und Parser lassen sich mit dem Pipe-Operator (`|`) direkt zu einer Kette verbinden.

In der Praxis bringt LangChain am meisten, wenn mehrere Schritte kombiniert werden müssen. Für einfache Einzelaufrufe ist der Overhead gering, aber unnötig.

---

- [Einführung](https://python.langchain.com/docs/introduction/)
- [Konzepte](https://python.langchain.com/docs/concepts/)
- [API-References](https://python.langchain.com/api_reference/index.html)
- [Integrations](https://python.langchain.com/docs/integrations/providers/)

---

- [Tutorials](https://python.langchain.com/docs/tutorials/)
- [How-to-Guides](https://python.langchain.com/docs/how_to/)

## 5.2 | Überblick Konzepte




Die Darstellung veranschaulicht das Grundprinzip von LangChain: die Verkettung verschiedener Konzepte zu einem durchgängigen Verarbeitungsprozess, der Eingaben in strukturierte Ausgaben umwandelt.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Verarbeitungskette</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    Input["Input<br/>(Message)"] --> Prompt

    subgraph Chain
        Prompt["Prompt<br/>(Template)"]
        Model["Model<br/>(LLM)"]
        Response["Response<br/>(Parser)"]
        Prompt --> Model --> Response
    end

    Response --> Result["Result"]
"""
mermaid(diagram)

LangChain baut auf wenigen Grundkonzepten auf, die in nahezu allen Notebooks wiederkehren:

**Messages** transportieren Inhalte zwischen Mensch und Modell — als System-Anweisung, User-Eingabe oder AI-Antwort. **Prompt-Templates** erzeugen diese Messages aus Vorlagen mit Platzhaltern. **Chat-Modelle** nehmen Nachrichten entgegen und geben Nachrichten zurück — das ist die Standard-Schnittstelle für alle neueren LLMs.

Für komplexere Anwendungen kommen **Chains** (verkettete Verarbeitungsschritte), **Structured Output** (typsichere Rückgaben per Pydantic), **Chat-History** (Konversationskontext), **RAG** (Retrieval-Augmented Generation), **Retriever** (Dokumentensuche), **Agents** (Tool-auswählende LLMs) und **Tools** (aufgerufene Funktionen) hinzu.

## 5.3 | Nachrichten-Typen

Beim Einsatz von Large Language Models (LLMs) wie GPT gibt es drei Hauptarten von Nachrichten (Messages), die die Interaktion mit dem Modell steuern. Diese Messages definieren, wie das Modell reagiert, welche Eingaben es erhält und welche Ausgaben es generiert.



In [ ]:
#@markdown   <p><font size="4" color='green'>  Nachrichten-Typen Workflow</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
graph TD
    A["System Message"] --> D["LLM Verarbeitung"]
    B["User Message"] --> D
    D --> C["AI Message"]

    A1["Definiert Verhalten<br/>Rolle & Persönlichkeit<br/>Einschränkungen"] -.-> A
    B1["Benutzer-Eingabe<br/>Frage oder Befehl<br/>Kontext"] -.-> B
    C1["Generierte Antwort<br/>Basierend auf System + User<br/>Strukturiertes Format"] -.-> C

    style A fill:#ffe1e1
    style B fill:#e1f5ff
    style C fill:#e1ffe1
    style D fill:#fff4e1
"""
mermaid(diagram, width=700, height=450)

**Kurz gefasst**   

+  **System Message** legt die Regeln und das Verhalten des Modells fest.
+ **User Message** stellt eine Anfrage oder einen Befehl.
+ **AI Message** gibt die generierte Antwort basierend auf den vorherigen Nachrichten.




## 5.4 | Ein einfaches Modell


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate


In [ ]:
# Modell definieren (Kurznotation: "provider:model")
llm = init_chat_model(BASELINE)

## 5.5 | Prompts

Im Diagramm aus Abschnitt 5.2 ist der Prompt der erste Schritt der Chain. Hier wird dieser Baustein als wiederverwendbare Vorlage mit Platzhaltern konkret.

Prompts sind die Eingaben an das LLM. LangChain bietet verschiedene Template-Systeme, deren Kernidee die **Vorlage mit Platzhaltern** ist: Ein Template definiert den Text einmalig, die Platzhalter `{placeholder}` werden zur Laufzeit mit konkreten Werten befüllt.

Templates machen Prompts wiederverwendbar — für viele Themen oder Fragen —, strukturiert durch Rollenaufteilung (System/user) und dynamisch erweiterbar durch Memory, Tools und Kontext. In der Praxis ist `ChatPromptTemplate` fast immer die richtige Wahl, sobald mehr als ein einfacher Freitext-Aufruf geplant ist.

Das **ChatPromptTemplate** ist eine spezielle Prompt-Vorlage für Chatmodelle (z. B. GPT-3.5, GPT-4), die **mehrere Rollen und Nachrichten** unterstützt – also genau das, was Chat-Modelle eigentlich brauchen.

Es gehört zur LangChain-Bibliothek und baut auf dem Prinzip auf:


**Wer sagt was? → system, user, assistant, etc.**

In LangChain unterstützte Rollen:

+ "system" - für Systemnachrichten
+ "assistant" oder "ai" - für Assistenten-/KI-Antworten
+ "user" oder "user" - für Benutzernachrichten

In [ ]:
# Template als Liste & Tupel
chat_template = ChatPromptTemplate([
    ("system", "{system_prompt}"),
    ("user", "Erkläre mir {thema}"),
])

In [ ]:
# Variablen zuweisen
system_prompt = "Du bist ein hilfreicher und humorvoller Assistent."
thema = "Machine Learning"

In [ ]:
# Template & Variablen kombinieren
prompt = chat_template.invoke({"system_prompt": system_prompt, "thema": thema})

In [ ]:
# Sprachmodell aufrufe
response = llm.invoke(prompt)

In [ ]:
# Ausgabe
mprint("## 📣 Model response:")
mprint("---")
mprint(response.content)

## 5.6 | Chain und Agent
---

Eine Chain führt festgelegte Verarbeitungsschritte in einer festen Reihenfolge aus. Ein Agent entscheidet dagegen anhand der Aufgabe und der verfügbaren Tool-Schemas, ob und welches Tool er aufruft.

**Warum diese Reihenfolge?**

M01 zeigt Chains nur als festen technischen Ablauf und Tools als kontrollierte Handlungsmöglichkeiten. M02 setzt darauf auf: Dort entscheidet erstmals ein Agent, ob er direkt antwortet oder ein Tool aufruft. Die Chain ist damit kein Gegenmodell zum Agenten, sondern der Baustein für festgelegte Teilschritte innerhalb eines Agentensystems.


<p><font color='black' size="5">
Zwei unterschiedliche Aufrufmuster
</font></p>




**Chain:** Der Entwickler definiert die Verarbeitungsschritte explizit.

```python
prompt = ChatPromptTemplate([
    ("system", "Du erklärst technische Begriffe kurz."),
    ("human", "{frage}"),
])

chain = prompt | llm
result = chain.invoke({"frage": "Was ist RAG?"})
```

> **Vorschau:** Dieses Agentenbeispiel dient hier nur dem Vergleich. Die ausführbare Implementierung folgt in **M02**, nachdem Tools und `create_agent()` eingeführt wurden.

**Agent:** Der Agent verwaltet Nachrichtenverlauf, Tool-Aufrufe und Entscheidungen.

```python
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Du bist ein hilfreicher Recherche-Agent.",
)

result = agent.invoke({
    "messages": [
        HumanMessage(content="Prüfe, ob RAG im Wissenskorpus vorkommt.")
    ]
})
```

> **Merksatz:** Chain = Der Entwickler definiert den Ablauf. Agent = Das Modell entscheidet innerhalb eines gesetzten Rahmens.



In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain_prompt = ChatPromptTemplate([
    ("system", "Du erklärst technische Begriffe kurz und verständlich."),
    ("human", "Erkläre {thema} in einem Satz."),
])

parser = StrOutputParser()

# Eine einfache Chain: Prompt → Modell → Textausgabe
simple_chain = chain_prompt | llm | parser
chain_result = simple_chain.invoke({"thema": "Tool-Nutzung"})
mprint(chain_result)


## 5.7 | Ausgabeformen: AIMessage und Text
---

Ein direkter Modellaufruf liefert typischerweise ein `AIMessage`-Objekt mit Metadaten. Für eine nachgelagerte Verarbeitung ist oft reiner Text praktischer. `StrOutputParser()` übernimmt diese Umwandlung als letzter Schritt einer Chain.

Die Details zu Parsern und weiteren Runnable-Bausteinen folgen in **M06**.


In [ ]:
message_result = llm.invoke("Nenne einen Vorteil von KI-Agenten in einem Satz.")
mprint(f"Rückgabetyp ohne Parser: {type(message_result).__name__}")
mprint(message_result.content)

text_chain = llm | StrOutputParser()
text_result = text_chain.invoke("Nenne einen zweiten Vorteil von KI-Agenten in einem Satz.")
mprint(f"Rückgabetyp mit Parser: {type(text_result).__name__}")
mprint(text_result)

## 5.8 | Vorschau: Strukturierte Ausgaben
---

Agenten müssen nicht immer Freitext liefern. Ein Schema kann festlegen, welche Felder eine Antwort enthalten soll. So wird die Ausgabe zu einem verlässlicheren Vertrag für nachgelagerte Schritte.

M01 zeigt nur das Prinzip. Pydantic-Schemas, Validierung und produktive Anwendungen behandelt **M04 Structured Output** ausführlich.


In [ ]:
from pydantic import BaseModel


class AgentenAntwort(BaseModel):
    antwort: str
    naechster_schritt: str


structured_llm = llm.with_structured_output(AgentenAntwort)
structured_result = structured_llm.invoke(
    "Formuliere eine kurze Antwort auf die Frage: Was ist ein KI-Agent? "
    "Nenne außerdem einen sinnvollen nächsten Lernschritt."
)
mprint(f"**Antwort:** {structured_result.antwort}")
mprint(f"**Nächster Schritt:** {structured_result.naechster_schritt}")


# 6 | Was sind Tools?
---



Bisher wurde betrachtet, wie eine Anfrage das Modell erreicht und in welcher Form es antwortet. Für Aufgaben mit aktuellen Daten oder Berechnungen braucht der Agent zusätzlich Werkzeuge. Kapitel 6 führt diese Handlungsmöglichkeiten ein; in M02 entscheidet der Agent selbst, wann er sie nutzt.

Tools sind Funktionen, die ein KI-Agent gezielt aufrufen kann – für Aufgaben, die das Sprachmodell allein nicht lösen kann: aktuelle Daten abfragen, rechnen, externe Systeme ansprechen.


Ein **Tool** ist eine Funktion, die ein Agent während seines ReAct-Zyklus aufrufen kann.

| Eigenschaft | Beschreibung |
|---|---|
| **Name** | Eindeutige Bezeichnung (= Funktionsname) |
| **Beschreibung** | Docstring – erklärt dem LLM, *wann* das Tool passt |
| **Schema** | Automatisch aus Typ-Hints generiert (JSON Schema) |
| **Rückgabewert** | String oder serialisierbarer Python-Typ |


Das LLM sieht **nur das Schema** (Name + Beschreibung + Parameter-Typen), niemals den eigentlichen Code. Damit ist das Schema ein **Agenten-Vertrag**: eine feste Schnittstelle, auf die sich Agent und Tool verlassen – ändert sich der Vertrag (z. B. ein Parametertyp), muss der Agent das zuverlässig erkennen können, statt am fehlerhaften Aufruf zu scheitern.

<p><font color='darkblue' size="4">💡 <b>Tipp</b></font></p>

Je klarer der Docstring, desto besser wählt der Agent das richtige Tool.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Kommunikationsdiagramm</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
sequenceDiagram
    autonumber
    participant U as Benutzer
    participant A as Agent (LLM)
    participant T as Tool

    U->>A: "Wie warm ist 37°C in Fahrenheit?"
    A->>A: Prüft verfügbare Tool-Schemas
    A->>T: celsius_nach_fahrenheit(temperatur=37.0)
    T->>T: Umrechnung °C auf °F
    T->>A: 98.6
    A->>U: "37 °C entsprechen 98,6° F"
'''

mermaid(diagram, width=750)

# 7 | @tool Decorator
---


Der `@tool`-Decorator aus `langchain_core.tools` verwandelt eine normale Python-Funktion in ein LangChain-Tool.

**Was passiert dabei automatisch:**
1. **Name** wird aus dem Funktionsnamen abgeleitet
2. **Beschreibung** wird aus dem Docstring übernommen
3. **Schema** wird aus den Typ-Hints als JSON Schema generiert

**Pflichtangaben für gute Tools:**
- ✅ Typ-Hints für alle Parameter und den Rückgabewert
- ✅ Klarer Docstring: was macht das Tool, was gibt es zurück?
- ✅ Sprechende Parameternamen

Das generierte Schema ist exakt das, was das LLM sieht, wenn es entscheidet, ob und wie es das Tool aufruft.

In [ ]:
import json
from langchain_core.tools import tool

@tool
def celsius_nach_fahrenheit(temperatur: float) -> float:
    """Rechnet eine Temperatur von Celsius in Fahrenheit um. Gibt den Fahrenheit-Wert zurück."""
    return round(temperatur * 9 / 5 + 32, 2)

# Schema anzeigen – was das LLM sieht
mprint("## Tool-Schema (was das LLM sieht)")
mprint(f"**Name:** `{celsius_nach_fahrenheit.name}`")
mprint(f"**Beschreibung:** {celsius_nach_fahrenheit.description}")
schema_json = json.dumps(celsius_nach_fahrenheit.args, indent=2, ensure_ascii=False)
mprint("**Parameter-Schema:**")
mprint("```json")
mprint(schema_json)
mprint("```")
mprint(f"**Funktion:** {celsius_nach_fahrenheit.func}")


# 8 | Weiteres Tool bauen
---


Zwei praxisnahe Tools werden isoliert getestet – ohne Agent.

**Tipp:** Tools immer zuerst isoliert testen, bevor diese in einen Agenten eingebunden werden. Das erleichtert das Debugging erheblich.

In [ ]:
from langchain_core.tools import tool

@tool
def ist_buerozeit(stunde: int) -> str:
    """Prüft ob eine Stunde des Tages (0–23) in die typische Bürozeit fällt (9–17 Uhr).
    Gibt 'Bürozeit' oder 'Außerhalb Bürozeit' zurück."""
    if 9 <= stunde <= 17:
        return "Bürozeit"
    return "Außerhalb Bürozeit"

# Alle verfügbaren Tools auflisten
tools = [celsius_nach_fahrenheit, ist_buerozeit]
mprint(f"**Definierte Tools:** `{[t.name for t in tools]}`")

In [ ]:
# .func() ruft die Python-Funktion direkt auf – kein Runnable, kein Tracing
mprint("## Tool-Tests")
mprint('---')
mprint(f"**37 °C → Fahrenheit:** {celsius_nach_fahrenheit.func(temperatur=37.0)} °F")
mprint(f"**100 °C → Fahrenheit:** {celsius_nach_fahrenheit.func(temperatur=100.0)} °F")
mprint(f"**0 °C → Fahrenheit:** {celsius_nach_fahrenheit.func(temperatur=0.0)} °F")
print()
mprint(f"**14 Uhr – Bürozeit?** {ist_buerozeit.func(stunde=14)}")
mprint(f"**20 Uhr – Bürozeit?** {ist_buerozeit.func(stunde=20)}")
mprint(f"**9 Uhr – Bürozeit?** {ist_buerozeit.func(stunde=9)}")

# 9 | Tool mit Fehlerbehandlung
---


Tools sollten Fehler **abfangen** oder bewusst eine Fehlerstrategie wählen – eine ungefangene Exception kann den Agent-Loop unterbrechen.

**Grundregeln für robuste Tools:**
- ✅ Eingaben validieren (Typen, Wertebereiche, erlaubte Werte)
- ✅ Fehler als informativen String zurückgeben, nicht als Exception
- ✅ Fehlermeldungen sollen dem LLM helfen, die Situation zu verstehen

Das LLM kann aus einer klaren Fehlermeldung ("Währung CHF nicht unterstützt, verfügbar: EUR, USD, GBP") selbst entscheiden, wie es weiter vorgeht.

In [ ]:
from langchain_core.tools import tool

@tool
def waehrung_umrechnen(betrag: float, von: str, nach: str) -> str:
    """Rechnet einen Geldbetrag zwischen EUR, USD und GBP um.
    Unterstützte Währungen: EUR, USD, GBP.
    Gibt das Ergebnis als formatierten String zurück."""
    kurse = {
        ("EUR", "USD"): 1.08, ("USD", "EUR"): 0.93,
        ("EUR", "GBP"): 0.85, ("GBP", "EUR"): 1.18,
        ("USD", "GBP"): 0.79, ("GBP", "USD"): 1.27,
    }
    von_norm = von.upper().strip()
    nach_norm = nach.upper().strip()

    if von_norm == nach_norm:
        return f"{betrag:.2f} {nach_norm} (keine Umrechnung nötig)"

    kurs = kurse.get((von_norm, nach_norm))
    if kurs is None:
        return (
            f"Fehler: Umrechnung von {von_norm} nach {nach_norm} nicht unterstützt. "
            f"Verfügbare Währungen: EUR, USD, GBP."
        )

    ergebnis = round(betrag * kurs, 2)
    return f"{betrag:.2f} {von_norm} = {ergebnis:.2f} {nach_norm} (Kurs: {kurs})"

In [ ]:
# .func() ruft die Python-Funktion direkt auf – kein Runnable, kein Tracing
mprint("## Fehlerbehandlung testen")
mprint(f"✅ Normal:              {waehrung_umrechnen.func(betrag=100.0, von='EUR', nach='USD')}")
mprint(f"✅ Gleiche Währung:     {waehrung_umrechnen.func(betrag=50.0, von='USD', nach='USD')}")
mprint(f"⚠️ Unbekannte Währung:  {waehrung_umrechnen.func(betrag=100.0, von='EUR', nach='CHF')}")


<p><font color='black' size="5">
Prüfstatus für Tool-Ergebnisse
</font></p>


Ein Agent darf ein Tool-Ergebnis nicht ungeprüft als Antwortgrundlage verwenden. Für M01 reicht ein kleiner, sichtbarer Prüfstatus als Grundlage für spätere Agenten-Gates.

In [ ]:
def bewerte_tool_ergebnis(ergebnis: str) -> str:
    """Ordnet ein Tool-Ergebnis für den nächsten Agentenschritt ein."""
    text = ergebnis.strip()
    if not text:
        return "nicht_im_korpus"
    if text.lower().startswith("fehler:"):
        return "unsicher"
    return "ausreichend"

for beispiel in ["RAG gefunden", "Fehler: keine Quelle", ""]:
    print(beispiel or "<leer>", "→", bewerte_tool_ergebnis(beispiel))

**Übergang zu M02 und M03**

M01 legt die Grundlagen: Tools werden definiert und geprüft, Chains zeigen feste Abläufe. In M02 übernimmt erstmals ein Agent die Toolwahl; M03 macht diese Entscheidungen über Prompts kontrollierbarer.

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Tools für den Meeting-Briefing-Agent
</font></p>

Der Meeting-Briefing-Agent benötigt kleine, klar beschriebene Werkzeuge: Begriffe normalisieren, Suchanfragen vorbereiten und Metadaten auswerten. Dieser Abschnitt legt dafür die Tool-Grundlage.

**Grundlagen**
- Mindestens zwei Tools mit dem `@tool`-Decorator definieren.
- Alle Parameter und Rückgabewerte mit Typ-Hints versehen.
- Tools direkt mit `.func(...)` testen.
- Tools in `meine_tools` speichern.

**✅ Erledigt wenn:** `meine_tools` enthält mindestens zwei Tools mit Namen, Docstring und Args-Schema.


In [ ]:
# Grundlagen: Tools für einfache Briefing-Agent-Hilfsfunktionen
from langchain_core.tools import tool

# 1. normalisiere_suchbegriff(begriff: str) -> str: Kleinschreibung + Whitespace normalisieren
# 2. zaehle_suchbegriffe(frage: str) -> int: Woerter > 2 Zeichen zaehlen
# 3. Beide Tools in meine_tools sammeln, mit .func(...) testen

**Aufbau**
- Mindestens ein Tool mit Fehlerbehandlung ergänzen.
- Ungültige Eingaben als verständlichen Fehlerstring zurückgeben.
- Den Fehlerfall direkt mit `.func(...)` testen.

**✅ Erledigt wenn:** Ein Fehlerfall liefert einen String mit `Fehler` statt einer Exception.


In [ ]:
# Aufbau: Fehlerbehandlung in einem Tool
# Startpunkt: meine_tools aus Grundlagen erweitern

# 1. berechne_begriffsdichte(text: str, begriff: str) -> str: Begriffsdichte berechnen
# 2. Leere Eingaben als "Fehler: ..."-String zurueckgeben, keine Exception werfen
# 3. Tool zu meine_tools hinzufuegen, Fehlerfall und Normalfall mit .func(...) testen

**Vertiefung**
- Ein drittes Tool mit Mock-Datenbank für Paper-Metadaten bauen.
- Ein zweites Tool-Argument mit Standardwert verwenden.

**✅ Erledigt wenn:** Die Mock-Abfrage liefert Paper-Metadaten und das Schema enthält den Standardwert.


In [ ]:
# Vertiefung: Mock-Datenbank und Args-Schema prüfen
import json

PAPER_DB = {
    "rag_survey": {"titel": "Retrieval-Augmented Generation Survey", "thema": "RAG-Architekturen", "jahr": 2023},
    "ragas": {"titel": "RAGAS: Automated Evaluation of Retrieval Augmented Generation", "thema": "Evaluation", "jahr": 2023},
}

# 1. suche_paper_metadaten(paper_id: str, format: str = "kurz") -> dict: Eintrag aus PAPER_DB liefern
# 2. Tool zu meine_tools hinzufuegen, mit .func(...) testen

# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Kursüberblick](https://ralf-42.github.io/Agenten/02-orientierung-entscheidung/kursueberblick.html)
- [Aufgabenklassen & Lösungswege](https://ralf-42.github.io/Agenten/02-orientierung-entscheidung/aufgabenklassen-und-loesungswege.html)
- [Terminologie](https://ralf-42.github.io/Agenten/02-orientierung-entscheidung/terminologie.html)
- [Meeting- & Research-Briefing-Agent](https://ralf-42.github.io/Agenten/02-orientierung-entscheidung/meeting-research-briefing-leitaufgabe.html)
- [Tool Use & Function Calling](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/tool-use-function-calling.html)
- [Agenten-Architekturen](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/agent-architekturen.html)